# 03 Búsqueda vectorial con TF-IDF
### David Alejandro Puga Novoa
#### Recuperación de información
#### 11 de Noviembre de 2026

### 1. Recuperar y revisar el corpus

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Construya una tabla con:

**Identificador de documento | Título | Número de términos únicos**

Presente:

- Número de documentos.
- Tamaño del vocabulario.
- Longitud mínima, media y máxima de los documentos.

Conserve el texto original para mostrar fragmentos de los resultados. Todas las técnicas deben utilizar **los mismos documentos y términos procesados**.

In [1]:
#Celda 1: carga y preprocesamiento
import re
import nltk
import pandas as pd
from nltk.corpus import gutenberg

nltk.download('gutenberg')

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-záéíóúüñ]+(?:'[a-z]+)*", text)

# Texto original y tokens procesados
doc_ids = gutenberg.fileids()
raw_docs = {d: gutenberg.raw(d) for d in doc_ids}
docs = {d: tokenize(t) for d, t in raw_docs.items()}

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\biqo\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [2]:
#Celda 2: títulos y tabla
titles = {
    "austen-emma.txt": "Emma",
    "austen-persuasion.txt": "Persuasion",
    "austen-sense.txt": "Sense and Sensibility",
    "bible-kjv.txt": "The King James Bible",
    "blake-poems.txt": "Poems (Blake)",
    "bryant-stories.txt": "Stories to Tell to Children",
    "burgess-busterbrown.txt": "The Adventures of Buster Bear",
    "carroll-alice.txt": "Alice's Adventures in Wonderland",
    "chesterton-ball.txt": "The Ball and the Cross",
    "chesterton-brown.txt": "The Wisdom of Father Brown",
    "chesterton-thursday.txt": "The Man Who Was Thursday",
    "edgeworth-parents.txt": "The Parent's Assistant",
    "melville-moby_dick.txt": "Moby Dick",
    "milton-paradise.txt": "Paradise Lost",
    "shakespeare-caesar.txt": "Julius Caesar",
    "shakespeare-hamlet.txt": "Hamlet",
    "shakespeare-macbeth.txt": "Macbeth",
    "whitman-leaves.txt": "Leaves of Grass",
}

tabla = pd.DataFrame({
    "Identificador": doc_ids,
    "Título": [titles.get(d, d) for d in doc_ids],
    "Términos únicos": [len(set(docs[d])) for d in doc_ids],
})
tabla

,Identificador,Título,Términos únicos
0,austen-emma.txt,Emma,7204
1,austen-persuasion.txt,Persuasion,5823
2,austen-sense.txt,Sense and Sensibility,6378
3,bible-kjv.txt,The King James Bible,12789
4,blake-poems.txt,Poems (Blake),1542
5,bryant-stories.txt,Stories to Tell to Children,3913
6,burgess-busterbrown.txt,The Adventures of Buster Bear,1543
7,carroll-alice.txt,Alice's Adventures in Wonderland,2629
8,chesterton-ball.txt,The Ball and the Cross,8283
9,chesterton-brown.txt,The Wisdom of Father Brown,7882


In [3]:
#Celda 3: estadísticas del corpus
vocab = sorted(set(t for toks in docs.values() for t in toks))
lengths = [len(toks) for toks in docs.values()]

print(f"Número de documentos: {len(docs)}")
print(f"Tamaño del vocabulario: {len(vocab)}")
print(f"Longitud mínima: {min(lengths)} tokens")
print(f"Longitud media: {sum(lengths)/len(lengths):.1f} tokens")
print(f"Longitud máxima: {max(lengths)} tokens")

Número de documentos: 18
Tamaño del vocabulario: 43494
Longitud mínima: 6837 tokens
Longitud media: 117717.2 tokens
Longitud máxima: 790052 tokens


### 2. Implementación de Funciones

Programe funciones para obtener:

1. TF de cada término en cada documento.
2. DF de cada término en **todo el corpus**.
3. IDF de cada término.
4. Los seis pesos TF-IDF correspondientes.

Para facilitar la comparación posterior con scikit-learn, utilizaremos:

$TF(t,d)=f(t,d)$

$
IDF(t)=\ln\left(\frac{N}{DF(t)}\right)+1
$

$
w_{t,d}=TF(t,d)\times IDF(t)
$

Aquí, $f(t,d)$ es el número de apariciones y $N$ el número de documentos.

**Nota:** Ahora utilizamos logaritmo natural y sumamos uno al IDF. Esta es la fórmula de scikit-learn cuando `smooth_idf=False`. [sklearn](https://sklearn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfTransformer.html?utm_source=chatgpt.com)

In [4]:
#Celda 1: funciones
import math
from collections import Counter

def compute_tf(tokens):
    """TF(t,d) = f(t,d): conteo de cada término en un documento."""
    return Counter(tokens)

def compute_df(docs):
    """DF(t): en cuántos documentos aparece t (cada documento cuenta una vez)."""
    df = Counter()
    for tokens in docs.values():
        df.update(set(tokens))
    return df

def compute_idf(df, N):
    """IDF(t) = ln(N / DF(t)) + 1"""
    return {t: math.log(N / c) + 1 for t, c in df.items()}

def compute_tfidf_weight(tf_value, idf_value):
    """w(t,d) = TF(t,d) * IDF(t)"""
    return tf_value * idf_value

In [5]:
#Celda 2: ejemplo de juguete (los seis pesos)
toy = {
    "d1": ["the", "cat", "sat", "on", "the", "mat"],
    "d2": ["the", "dog", "sat", "on", "the", "log"],
    "d3": ["the", "cat", "chased", "the", "dog"],
}
toy_terms = ["cat", "chased"]

N_toy = len(toy)
df_toy = compute_df(toy)
idf_toy = compute_idf(df_toy, N_toy)

print(f"N = {N_toy}")
for t in toy_terms:
    print(f"DF({t}) = {df_toy[t]}  ->  IDF({t}) = ln({N_toy}/{df_toy[t]}) + 1 = {idf_toy[t]:.6f}")

print(f"\n{'Término':<10}{'Doc':<6}{'TF':>4}{'IDF':>10}{'TF-IDF':>10}")
for t in toy_terms:
    for d, tokens in toy.items():
        f = compute_tf(tokens)[t]
        print(f"{t:<10}{d:<6}{f:>4}{idf_toy[t]:>10.6f}{compute_tfidf_weight(f, idf_toy[t]):>10.6f}")

N = 3
DF(cat) = 2  ->  IDF(cat) = ln(3/2) + 1 = 1.405465
DF(chased) = 1  ->  IDF(chased) = ln(3/1) + 1 = 2.098612

Término   Doc     TF       IDF    TF-IDF
cat       d1       1  1.405465  1.405465
cat       d2       0  1.405465  0.000000
cat       d3       1  1.405465  1.405465
chased    d1       0  2.098612  0.000000
chased    d2       0  2.098612  0.000000
chased    d3       1  2.098612  2.098612


### 3. Implementar TF-IDF para el corpus

1. Construir un vocabulario con un orden definido.
2. Contar las frecuencias TF.
3. Calcular DF contando cada documento una sola vez por término.
4. Calcular IDF con la función de la sección anterior.
5. Obtener la matriz documento-término TF-IDF.

In [6]:
#Celda 1: construcción de la matriz
import numpy as np

# 1. Vocabulario con orden definido (alfabético) y su índice
vocab = sorted(set(t for toks in docs.values() for t in toks))
term_index = {t: i for i, t in enumerate(vocab)}
doc_list = list(docs.keys())          # orden fijo de filas
N = len(doc_list)

# 2. Matriz de frecuencias TF (N x V)
tf_matrix = np.zeros((N, len(vocab)))
for i, d in enumerate(doc_list):
    for term, f in compute_tf(docs[d]).items():
        tf_matrix[i, term_index[term]] = f

# 3. DF contando cada documento una sola vez por término
df_counter = compute_df(docs)
df_vec = np.array([df_counter[t] for t in vocab])

# 4. IDF con la función de la sección 2
idf_dict = compute_idf(df_counter, N)
idf_vec = np.array([idf_dict[t] for t in vocab])

# 5. Matriz TF-IDF: w(t,d) = TF(t,d) * IDF(t)
tfidf_matrix = tf_matrix * idf_vec

print("Forma de la matriz (docs x vocabulario):", tfidf_matrix.shape)
print("Memoria:", round(tfidf_matrix.nbytes / 1e6, 1), "MB")

Forma de la matriz (docs x vocabulario): (18, 43494)
Memoria: 6.3 MB


In [7]:
#Celda 2: comprobaciones
# Sumas de TF por documento = longitud del documento
print("TF suma == longitud:", all(tf_matrix[i].sum() == len(docs[d]) for i, d in enumerate(doc_list)))

# Términos con DF = N tienen el IDF mínimo posible: ln(1) + 1 = 1
print("DF('the') =", df_vec[term_index['the']], "| IDF =", idf_vec[term_index['the']])

# Un término raro (DF = 1) tiene el IDF máximo: ln(18) + 1
print("IDF máximo esperado:", round(math.log(N) + 1, 6), "| obtenido:", idf_vec.max().round(6))

# Ejemplo: top-5 pesos TF-IDF de Moby Dick
i = doc_list.index("melville-moby_dick.txt")
top = np.argsort(tfidf_matrix[i])[::-1][:5]
for j in top:
    print(f"{vocab[j]:<12} TF={tf_matrix[i, j]:.0f}  IDF={idf_vec[j]:.4f}  TF-IDF={tfidf_matrix[i, j]:.2f}")

TF suma == longitud: True
DF('the') = 18 | IDF = 1.0
IDF máximo esperado: 3.890372 | obtenido: 3.890372
the          TF=14431  IDF=1.0000  TF-IDF=14431.00
of           TF=6609  IDF=1.0000  TF-IDF=6609.00
and          TF=6430  IDF=1.0000  TF-IDF=6430.00
a            TF=4728  IDF=1.0000  TF-IDF=4728.00
to           TF=4625  IDF=1.0000  TF-IDF=4625.00


### 4. Verificar con `TfidfVectorizer`

Construya una segunda representación utilizando la librería `TfidfVectorizer`, con esta configuración:

| Parámetro | Valor |
|---|---|
| `use_idf` | `True` |
| `smooth_idf` | `False` |
| `sublinear_tf` | `False` |
| `norm` | `None` |

Esta configuración permite comparar pesos sin normalización y con TF como conteo directo. [scikit-learn 1.9.1 documentation](https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html?highlight=vectorizer\&utm_source=chatgpt.com)

Aseguren que el vectorizador recibe exactamente los mismos tokens que su implementación.

Calcule la **máxima diferencia absoluta** entre ambas matrices.

Si no coinciden, investiguen el preprocesamiento, el orden de columnas, la fórmula de IDF y la normalización.

In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    analyzer=lambda tokens: tokens,   # usa los tokens ya procesados tal cual
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None,
)

# Mismo orden de filas que doc_list
X_sklearn = vectorizer.fit_transform([docs[d] for d in doc_list])
print("Forma:", X_sklearn.shape)

# 1. Mismo orden de columnas
feature_names = vectorizer.get_feature_names_out()
print("Mismo orden de columnas:", list(feature_names) == vocab)

# 2. Máxima diferencia absoluta entre ambas matrices
X_dense = X_sklearn.toarray()
max_diff = np.abs(tfidf_matrix - X_dense).max()
print("Máxima diferencia absoluta:", max_diff)
print("np.allclose:", np.allclose(tfidf_matrix, X_dense, atol=1e-9))

Forma: (18, 43494)
Mismo orden de columnas: True
Máxima diferencia absoluta: 0.0
np.allclose: True


### 5. Representar consultas y calcular similitud coseno

Defina **cuatro consultas** que representen necesidades de información diferentes.

Para cada consulta:

1. Escriban qué información espera encontrar el usuario.
2. Apliquen el mismo preprocesamiento de los documentos.
3. Construyan su vector TF-IDF utilizando **el vocabulario y los IDF del corpus**.
4. No recalculen IDF incorporando la consulta como otro documento.

Utilice `transform` del vectorizador ajustado.

**Similitud coseno**

La similitud coseno compara la orientación de los vectores de consulta y documento:

$
\operatorname{sim}(q,d)
=\frac{q\cdot d}{\|q\|\|d\|}
=\frac{\sum_i q_i d_i}
{\sqrt{\sum_i q_i^2}\sqrt{\sum_i d_i^2}}
$

Implemente este cálculo mediante el producto punto y las magnitudes de los vectores. Luego, comparen los resultados con `sklearn.metrics.pairwise.cosine_similarity`, que calcula ese producto punto normalizado. [sklearn](https://sklearn.org/stable/modules/metrics.html?utm_source=chatgpt.com)

Para una combinación consulta-documento, muestren:

**Términos compartidos | Productos parciales | Producto punto | Normas | Similitud**

In [9]:
#Celda 1: consultas y su necesidad de información
queries = {
    "q1": ("white whale captain ship sea",
           "Relatos de caza de ballenas y travesías en barco."),
    "q2": ("marriage love family estate",
           "Novelas de costumbres sobre matrimonio, familia y sociedad."),
    "q3": ("ghost murder king crown",
           "Tragedias sobre asesinatos, fantasmas y poder real."),
    "q4": ("satan paradise heaven fall",
           "Textos religiosos o épicos sobre el cielo, la caída y Satán."),
}

for k, (texto, necesidad) in queries.items():
    print(f"{k}: '{texto}'\n    Se espera: {necesidad}")

q1: 'white whale captain ship sea'
    Se espera: Relatos de caza de ballenas y travesías en barco.
q2: 'marriage love family estate'
    Se espera: Novelas de costumbres sobre matrimonio, familia y sociedad.
q3: 'ghost murder king crown'
    Se espera: Tragedias sobre asesinatos, fantasmas y poder real.
q4: 'satan paradise heaven fall'
    Se espera: Textos religiosos o épicos sobre el cielo, la caída y Satán.


In [10]:
#Celda 2: vector de la consulta con el vocabulario y los IDF del corpus
def query_vector(text):
    """TF-IDF de la consulta usando el vocabulario y el IDF del corpus (sin recalcular IDF)."""
    tokens = tokenize(text)                      # mismo preprocesamiento
    q = np.zeros(len(vocab))
    for t, f in compute_tf(tokens).items():
        if t in term_index:                      # términos fuera del vocabulario se ignoran
            q[term_index[t]] = f * idf_vec[term_index[t]]
    return q

q_vecs = {k: query_vector(texto) for k, (texto, _) in queries.items()}

# Verificación contra transform() del vectorizador ya ajustado
for k, (texto, _) in queries.items():
    q_sk = vectorizer.transform([tokenize(texto)]).toarray()[0]
    fuera = [t for t in tokenize(texto) if t not in term_index]
    print(f"{k}: diff máx = {np.abs(q_vecs[k] - q_sk).max()} | fuera del vocabulario: {fuera}")

q1: diff máx = 0.0 | fuera del vocabulario: []
q2: diff máx = 0.0 | fuera del vocabulario: []
q3: diff máx = 0.0 | fuera del vocabulario: []
q4: diff máx = 0.0 | fuera del vocabulario: []


In [11]:
#Celda 3: similitud coseno con producto punto, comparada con scikit-learn
from sklearn.metrics.pairwise import cosine_similarity

doc_norms = np.linalg.norm(tfidf_matrix, axis=1)

def cosine_scores(q):
    """sim(q,d) = (q·d) / (||q|| ||d||) para todos los documentos."""
    dots = tfidf_matrix @ q
    denom = doc_norms * np.linalg.norm(q)
    return np.divide(dots, denom, out=np.zeros_like(dots), where=denom > 0)

scores_manual = {k: cosine_scores(q) for k, q in q_vecs.items()}

for k, q in q_vecs.items():
    s_sk = cosine_similarity(q.reshape(1, -1), tfidf_matrix)[0]
    print(f"{k}: diff máx manual vs sklearn = {np.abs(scores_manual[k] - s_sk).max():.2e}")

q1: diff máx manual vs sklearn = 1.67e-16
q2: diff máx manual vs sklearn = 3.99e-17
q3: diff máx manual vs sklearn = 9.02e-17
q4: diff máx manual vs sklearn = 2.50e-16


In [12]:
#Celda 4: desglose para una combinación consulta-documento
def desglose(qk, doc):
    q = q_vecs[qk]
    i = doc_list.index(doc)
    d = tfidf_matrix[i]
    comunes = [j for j in np.nonzero(q)[0] if d[j] > 0]

    print(f"Consulta {qk}: '{queries[qk][0]}'  |  Documento: {doc}\n")
    print(f"{'Término compartido':<20}{'q_i':>10}{'d_i':>12}{'q_i·d_i':>14}")
    for j in comunes:
        print(f"{vocab[j]:<20}{q[j]:>10.4f}{d[j]:>12.4f}{q[j]*d[j]:>14.4f}")

    dot = float(q @ d)
    nq, nd = np.linalg.norm(q), np.linalg.norm(d)
    print(f"\nProducto punto = {dot:.4f}")
    print(f"||q|| = {nq:.4f}  |  ||d|| = {nd:.4f}")
    print(f"Similitud = {dot:.4f} / ({nq:.4f} × {nd:.4f}) = {dot/(nq*nd):.6f}")
    print(f"Verificación (vector completo): {scores_manual[qk][i]:.6f}")

desglose("q1", "melville-moby_dick.txt")

Consulta q1: 'white whale captain ship sea'  |  Documento: melville-moby_dick.txt

Término compartido         q_i         d_i       q_i·d_i
captain                 1.6931    521.4893      882.9582
sea                     1.1178    507.4735      567.2453
ship                    1.3254    580.5350      769.4541
whale                   2.0986   2310.5721     4848.9951
white                   1.1178    314.0970      351.0923

Producto punto = 7419.7450
||q|| = 3.3951  |  ||d|| = 21338.6420
Similitud = 7419.7450 / (3.3951 × 21338.6420) = 0.102417
Verificación (vector completo): 0.102417


### 6. Construir el ranking usando TF-IDF

Para cada consulta, ordenen los documentos de mayor a menor similitud coseno y presenten el **top-5**:

**Posición | Identificador | Título | Puntuación | Fragmento**

Las versiones explícita y basada en bibliotecas deben producir las mismas puntuaciones dentro de la tolerancia establecida.

In [13]:
#Celda 1: ranking con fragmentos
pd.set_option("display.max_colwidth", None)

def fragmento(doc, query_text, width=80):
    raw = raw_docs[doc]
    i = doc_list.index(doc)
    terms = [t for t in tokenize(query_text) if t in term_index and tfidf_matrix[i, term_index[t]] > 0]
    if not terms:
        return ""
    best = max(terms, key=lambda t: tfidf_matrix[i, term_index[t]])
    m = re.search(r"\b" + re.escape(best) + r"\b", raw, flags=re.IGNORECASE)
    if not m:
        return ""
    s, e = max(0, m.start() - width), min(len(raw), m.end() + width)
    return f"[{best}] ..." + re.sub(r"\s+", " ", raw[s:e]) + "..."

def ranking(qk, scores, k=5):
    order = np.argsort(-scores)[:k]
    rows = []
    for pos, i in enumerate(order, 1):
        d = doc_list[i]
        rows.append({
            "Posición": pos,
            "Identificador": d,
            "Título": titles[d],
            "Puntuación": round(float(scores[i]), 6),
            "Fragmento": fragmento(d, queries[qk][0]),
        })
    return pd.DataFrame(rows)

for qk, (texto, necesidad) in queries.items():
    print(f"\n=== {qk}: '{texto}' ===")
    display(ranking(qk, scores_manual[qk]))


=== q1: 'white whale captain ship sea' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,melville-moby_dick.txt,Moby Dick,0.102417,"[whale] ... ""While you take in hand to school others, and to teach them by what name a whale-fish is to be called in our tongue leaving out, through ignorance, the letter ..."
1,2,austen-persuasion.txt,Persuasion,0.033563,"[captain] ...th, the former curate of Monkford, however suspicious appearances may be, but a Captain Frederick Wentworth, his brother, who being made commander in consequence of th..."
2,3,whitman-leaves.txt,Leaves of Grass,0.011832,"[sea] ...chant of battles, I above all promote brave soldiers. } In Cabin'd Ships at Sea In cabin'd ships at sea, The boundless blue on every side expanding, With whis..."
3,4,bryant-stories.txt,Stories to Tell to Children,0.009480,"[white] ... were asleep, and nobody at all looked at Little Jack Rollaround except an old White Owl; and all she said was, ""Who are you?"" The little boy did not like her, ..."
4,5,chesterton-brown.txt,The Wisdom of Father Brown,0.009175,"[sea] ... eminent criminologist and specialist in certain moral disorders, lay along the sea-front at Scarborough, in a series of very large and well-lighted french windows..."



=== q2: 'marriage love family estate' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,blake-poems.txt,Poems (Blake),0.022360,"[love] ...d we are put on earth a little space, That we may learn to bear the beams of love And these black bodies and this sunburnt face Is but a cloud, and like a sh..."
1,2,austen-sense.txt,Sense and Sensibility,0.012456,"[family] ...[Sense and Sensibility by Jane Austen 1811] CHAPTER 1 The family of Dashwood had long been settled in Sussex. Their estate was large, and their ..."
2,3,austen-persuasion.txt,Persuasion,0.012400,"[family] ...ut Sir Walter had improved it by adding, for the information of himself and his family, these words, after the date of Mary's birth-- ""Married, December 16, 1810, Cha..."
3,4,whitman-leaves.txt,Leaves of Grass,0.011063,"[love] ...faith, Consort to every ship that sails, sail you! Bear forth to them folded my love, (dear mariners, for you I fold it here in every leaf;) Speed on my book! s..."
4,5,austen-emma.txt,Emma,0.009304,"[love] ... him to Miss Churchill, of a great Yorkshire family, and Miss Churchill fell in love with him, nobody was surprized, except her brother and his wife, who had never ..."



=== q3: 'ghost murder king crown' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,shakespeare-hamlet.txt,Hamlet,0.038672,[king] ... there? Fran. Nay answer me: Stand & vnfold your selfe Bar. Long liue the King Fran. Barnardo? Bar. He Fran. You come most carefully vpon your houre...
1,2,shakespeare-macbeth.txt,Macbeth,0.017720,"[king] ...ugh the fogge and filthie ayre. Exeunt. Scena Secunda. Alarum within. Enter King Malcome, Donalbaine, Lenox, with attendants, meeting a bleeding Captaine. Ki..."
2,3,bryant-stories.txt,Stories to Tell to Children,0.012606,"[king] ...eat big jungle; and in the jungle there was a great big Lion; and the Lion was king of the jungle. Whenever he wanted anything to eat, all he had to do was to com..."
3,4,bible-kjv.txt,The King James Bible,0.011897,[king] ...[The King James Bible] The Old Testament of the King James Bible The First Book of Mose...
4,5,carroll-alice.txt,Alice's Adventures in Wonderland,0.010820,"[king] ...d went by without noticing her. Then followed the Knave of Hearts, carrying the King's crown on a crimson velvet cushion; and, last of all this grand procession, ca..."



=== q4: 'satan paradise heaven fall' ===


,Posición,Identificador,Título,Puntuación,Fragmento
0,1,milton-paradise.txt,Paradise Lost,0.053203,"[heaven] ...sert Eternal Providence, And justify the ways of God to men. Say first--for Heaven hides nothing from thy view, Nor the deep tract of Hell--say first what cause ..."
1,2,blake-poems.txt,Poems (Blake),0.006397,"[heaven] ...on me, on all, Who became an infant small; Infant smiles are his own smiles; Heaven and earth to peace beguiles. DIVINE IMAGE To Mercy, Pity, Peace, and Lo..."
2,3,bible-kjv.txt,The King James Bible,0.004611,"[heaven] ...The First Book of Moses: Called Genesis 1:1 In the beginning God created the heaven and the earth. 1:2 And the earth was without form, and void; and darkness was ..."
3,4,chesterton-ball.txt,The Ball and the Cross,0.004266,"[heaven] ... since our science has spoken, the bottom has fallen out of the Universe. Now, heaven is the hopeless thing, more hopeless than any hell. Now, if there be any comfo..."
4,5,chesterton-thursday.txt,The Man Who Was Thursday,0.003002,[heaven] ...that place for its strange sunset. It looked like the end of the world. All the heaven seemed covered with a quite vivid and palpable plumage; you could only say that...


In [14]:
#Celda 2: comparar explícito vs scikit-learn
TOL = 1e-9

for qk, (texto, _) in queries.items():
    q_sk = vectorizer.transform([tokenize(texto)])
    s_sk = cosine_similarity(q_sk, X_sklearn)[0]
    s_man = scores_manual[qk]

    mismas_puntuaciones = np.allclose(s_man, s_sk, atol=TOL)
    mismo_top5 = list(np.argsort(-s_man)[:5]) == list(np.argsort(-s_sk)[:5])
    print(f"{qk}: diff máx = {np.abs(s_man - s_sk).max():.2e} | "
          f"puntuaciones iguales (tol {TOL}): {mismas_puntuaciones} | mismo top-5: {mismo_top5}")

q1: diff máx = 1.12e-15 | puntuaciones iguales (tol 1e-09): True | mismo top-5: True
q2: diff máx = 1.05e-15 | puntuaciones iguales (tol 1e-09): True | mismo top-5: True
q3: diff máx = 6.31e-16 | puntuaciones iguales (tol 1e-09): True | mismo top-5: True
q4: diff máx = 5.76e-16 | puntuaciones iguales (tol 1e-09): True | mismo top-5: True


## Conclusiones

**Corpus y preprocesamiento.** Se trabajó con los 18 documentos de Gutenberg de NLTK (vocabulario de 43.494 términos). Las longitudes van de 6.837 a 790.052 tokens, con una media de 117.717,2. Esa enorme variación de longitud condiciona varios resultados posteriores, porque TF se usó como conteo directo, sin normalizar por longitud.

**Implementación y verificación del TF-IDF.** La implementación explícita (TF = f(t,d), IDF = ln(N/DF) + 1, w = TF × IDF) coincide con `TfidfVectorizer` (`smooth_idf=False`, `sublinear_tf=False`, `norm=None`), con una diferencia máxima absoluta de 0,0 y el mismo orden de columnas. Esto confirma que la fórmula y el preprocesamiento son los correctos. Para garantizar que ambas versiones recibieran exactamente los mismos tokens, se pasó al vectorizador las listas ya procesadas (`analyzer` personalizado).

**Comportamiento del IDF.** Con N = 18, el IDF toma valores entre 1 (términos presentes en los 18 documentos, como `the`) y ln(18) + 1 ≈ 3,890 (términos presentes en un solo documento). El IDF penaliza los términos comunes y premia los distintivos. Sin embargo, como TF no está normalizado, las palabras muy frecuentes siguen dominando los vectores de los documentos largos: en *Moby Dick*, `the`, `of` y `and` tienen los mayores pesos (14.431, 6.609 y 6.430), aunque su IDF sea el mínimo.

**Rankings.** Para las cuatro consultas, las versiones explícita y de scikit-learn produjeron las mismas puntuaciones (diferencia máxima de 1,12e-15, tolerancia 1e-9) y el mismo top-5. Los resultados fueron coherentes con las necesidades de información: *Moby Dick* lidera la consulta sobre ballenas (0,102), *Hamlet* y *Macbeth* las dos primeras posiciones de la consulta sobre reyes y asesinatos, y *Paradise Lost* la consulta sobre el paraíso y la caída (0,053).

**Limitaciones observadas.**
- En la consulta sobre matrimonio y familia, *Poems* (Blake) queda primero por encima de las novelas de Austen. Es un documento corto en el que `love` pesa mucho dentro de su vector, mientras que en las novelas largas ese peso se diluye. Esto muestra el efecto de la longitud de los documentos sobre la similitud coseno con TF sin normalizar.
- En varias consultas, las posiciones 3 a 5 tienen puntuaciones muy bajas (≈ 0,003 a 0,012), lo que indica coincidencias parciales y no documentos realmente relevantes.
- La evaluación es cualitativa: solo se usaron cuatro consultas y no se dispone de juicios de relevancia, por lo que no se puede medir precisión ni recall.